# Memory hygiene grid on a Kaggle GPU

Runs the full experiment grid on a free Kaggle GPU (about 10-30x faster than a laptop CPU).

**Before you start (Kaggle sidebar, right):**
1. *Session options > Accelerator*: **GPU T4 x2** (or P100).
2. *Session options > Internet*: **On** (needs a phone-verified Kaggle account). The datasets and models are downloaded from GitHub and Hugging Face.
3. Get the code in, either way:
   * **A (recommended):** commit and push the code to GitHub, then set `REPO_URL` below. A private repo needs a GitHub token in *Add-ons > Secrets* named `GITHUB_TOKEN`.
   * **B:** zip the `LLM-Memory` folder, upload it with *Add Input > Upload > New Dataset*, and set `REPO_URL = None`.

Then *Run All*. Results are written to `/kaggle/working/` and zipped at the end, so you can download them from the *Output* tab.

In [ ]:
REPO_URL = "https://github.com/anjaligundawar/LLM-Memory.git"   # set to None to use an uploaded dataset instead
BRANCH = "main"
CORPORA = ["locomo", "clutrr", "loghub", "mab_conflict", "longmemeval"]   # run order; drop any you don't need
CONFIG = "configs/main.yaml"

In [ ]:
import os, subprocess, glob, shutil
os.chdir("/kaggle/working")
if REPO_URL:
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass  # public repo, no token needed
    if not os.path.exists("LLM-Memory"):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "LLM-Memory"], check=True)
else:
    src = next(p for p in glob.glob("/kaggle/input/*/**/requirements.txt", recursive=True))
    shutil.copytree(os.path.dirname(src), "LLM-Memory", dirs_exist_ok=True)
os.chdir("/kaggle/working/LLM-Memory")
print(os.getcwd(), os.listdir())

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Quick check that everything works (about a minute).

In [ ]:
!python -m pytest -q tests
!python scripts/download_data.py

The main grid. Each finished setting is saved immediately, so if the session stops you can re-run this cell
in a new session (after re-uploading `results` and `cache`) and it continues where it left off.
Kaggle sessions last up to 12 hours.

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text())
cfg["embedder"]["device"] = "cuda"
cfg["embedder"]["batch_size"] = 256
cfg.setdefault("nli", {})["device"] = "cuda"
cfg["nli"]["batch_size"] = 128
pathlib.Path("configs/kaggle.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
for c in CORPORA:
    !python scripts/run_grid.py configs/kaggle.yaml --corpora {c}

In [ ]:
!python scripts/make_tables.py results/main
!python scripts/make_figures.py results/main

Zip results (and the cache, so a later run or a laptop run can reuse every embedding and NLI score). Download from the *Output* tab.

In [ ]:
shutil.make_archive("/kaggle/working/results_main", "zip", "results/main")
shutil.make_archive("/kaggle/working/cache", "zip", "cache")
print(os.listdir("/kaggle/working"))